# 다음 카페 게시판 크롤링
- 대상 게시판: https://cafe.daum.net/skfootball/IxV3
- articles.push() JS 변수 파싱 방식 (1페이지부터 수집)

In [ ]:
import re
import json
import time
import requests
from datetime import datetime
import pandas as pd

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://cafe.daum.net/skfootball/IxV3"
}

BASE_URL = "https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3"

In [ ]:
def _parse_pushes(raw):
    """articles.push(...) 블록에서 dataid / title / created 추출"""
    pushes = re.findall(r'articles\.push\(\{[^}]+\}\)', raw)
    result = []
    for item in pushes:
        dm = re.search(r"dataid:\s*'(\d+)'", item)
        tm = re.search(r"title:\s*'([^']*)'" , item)
        cm = re.search(r"created:\s*'([^']+)'", item)
        if dm and tm:
            created_str = cm.group(1) if cm else None
            created_dt  = datetime.strptime(created_str, '%y.%m.%d') if created_str else None
            result.append({
                "dataid" : dm.group(1),
                "title"  : json.loads(f'"{ tm.group(1)}"'),
                "created": created_dt,
            })
    return result


def _get_page(page_num, prev_page, first_depth, last_depth):
    url = (
        f"https://cafe.daum.net/_c21_/bbs_list?grpid=1O7ju&fldid=IxV3"
        f"&page={page_num}&prev_page={prev_page}&listnum=20"
        f"&firstbbsdepth={first_depth}&lastbbsdepth={last_depth}"
    )
    raw = requests.get(url, headers=headers).text
    new_first = re.search(r"firstBbsDepth:\s*'([^']+)'", raw)
    new_last  = re.search(r"lastBbsDepth:\s*'([^']+)'",  raw)
    return (
        _parse_pushes(raw),
        page_num,
        new_first.group(1) if new_first else None,
        new_last.group(1)  if new_last  else None,
    )


def collect_posts(max_pages=4, date_from=None, date_to=None):
    """
    게시글 제목 + 작성일 수집 (1페이지부터 시작).

    Parameters
    ----------
    max_pages : 수집할 최대 페이지 수 (페이지당 20개)
    date_from : 시작일 'YYYY-MM-DD' 문자열 (None이면 제한 없음)
    date_to   : 종료일 'YYYY-MM-DD' 문자열 (None이면 제한 없음)

    Returns
    -------
    list of dict  {"dataid", "title", "created"}
    """
    from_dt = datetime.strptime(date_from, '%Y-%m-%d') if date_from else None
    to_dt   = datetime.strptime(date_to,   '%Y-%m-%d') if date_to   else None

    # 1페이지
    raw0 = requests.get(BASE_URL, headers=headers).text
    all_posts = _parse_pushes(raw0)
    cur_first = re.search(r"firstBbsDepth:\s*'([^']+)'", raw0).group(1)
    cur_last  = re.search(r"lastBbsDepth:\s*'([^']+)'",  raw0).group(1)
    cur_page  = 1
    print(f"page 1 — {len(all_posts)}개 수집")

    # 2페이지~
    for _ in range(max_pages - 1):
        time.sleep(1)
        posts, cur_page, cur_first, cur_last = _get_page(
            cur_page + 1, cur_page, cur_first, cur_last
        )
        print(f"page {cur_page} — {len(posts)}개 수집")
        all_posts.extend(posts)
        if not cur_first:
            print("마지막 페이지 도달")
            break

    print(f"\n총 수집: {len(all_posts)}개")

    if from_dt or to_dt:
        all_posts = [
            p for p in all_posts
            if p["created"] is not None
            and (from_dt is None or p["created"] >= from_dt)
            and (to_dt   is None or p["created"] <= to_dt)
        ]
        print(f"날짜 필터 적용 ({date_from} ~ {date_to}): {len(all_posts)}개")

    return all_posts

In [ ]:
# 전체 수집:   collect_posts(max_pages=5)
# 기간 지정:   collect_posts(max_pages=10, date_from="2026-05-01", date_to="2026-05-22")
posts = collect_posts(max_pages=5)

df = pd.DataFrame(posts)
df